和crocoddyl自带的的数值微分对比，看符号微分是否算对

### setup

In [ ]:
import os
import signal
import sys
import time

import numpy as np
import pinocchio as pin
import pinocchio.visualize

import crocoddyl

In [ ]:
WITHDISPLAY = "display"

# Loading the a1 model from local assets
urdf_path = "../assets/unitree_a1/urdf/a1.urdf"
srdf_path = "../assets/unitree_a1/srdf/a1.srdf"

rmodel = pin.buildModelFromUrdf(urdf_path, pin.JointModelFreeFlyer())
pin.loadReferenceConfigurations(rmodel, srdf_path, False)

lims = rmodel.effortLimit
lims *= 0.5  # reduced artificially the torque limits
rmodel.effortLimit = lims

lfFoot, rfFoot, lhFoot, rhFoot = "FL_foot", "FR_foot", "RL_foot", "RR_foot"
lfCalf, rfCalf, lhCalf, rhCalf = "FL_calf", "FR_calf", "RL_calf", "RR_calf"
body = "trunk"

rdata = rmodel.createData()
state = crocoddyl.StateMultibody(rmodel)
actuation = crocoddyl.ActuationModelFloatingBase(state)
lf_id = rmodel.getFrameId(lfFoot)
rf_id = rmodel.getFrameId(rfFoot)
lh_id = rmodel.getFrameId(lhFoot)
rh_id = rmodel.getFrameId(rhFoot)
body_id = rmodel.getFrameId(body)
lfcalf_id = rmodel.getFrameId(lfCalf)
rfcalf_id = rmodel.getFrameId(rfCalf)
lhcalf_id = rmodel.getFrameId(lhCalf)
rhcalf_id = rmodel.getFrameId(rhCalf)

_integrator = 'euler'
_control = 'one'

q0 = rmodel.referenceConfigurations["standing"]
rmodel.defaultState = np.concatenate([q0, np.zeros(rmodel.nv)])
x0 = rmodel.defaultState
u0 = np.zeros(actuation.nu)+1

In [8]:
from utils.costs import handstand
# 考虑一次one shoot
costs = handstand(rmodel, state, actuation)

AttributeError: module 'hppfcl' has no attribute 'hppfcl'

### test 1: 静止, u=0

In [ ]:
import utils.models
from utils.models import DAM_contact, ContactModel

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0)
x0 = rmodel.defaultState

u0 = np.ones(12)*0

DAD = DAM.createData()

# 符号微分
DAM.calc(DAD,x0,u0)
DAM.calcDiff(DAD,x0,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [ ]:
# 使用crocoddyl的数值微分
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x0, u0)
DAMND.calcDiff(DADND, x0, u0)
NDFx = DADND.Fx
NDFu = DADND.Fu

In [130]:
np.allclose(myFx[:,18:],NDFx[:,18:],atol=1e-5)

True

In [131]:
np.allclose(myFx[:,:18],NDFx[:,:18],atol=1e-5)

True

In [132]:
np.allclose(myFu,NDFu,atol=1e-5)

True

### test 2: slide+separate, u=1, 有slide梯度部分公式是错的, Es求不了导

In [ ]:
import utils.models
import warnings
warnings.filterwarnings('ignore')
import importlib
importlib.reload(utils.models)
from utils.models import DAM_contact

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0)
x0 = rmodel.defaultState

u0 = np.ones(12)*1

DAD = DAM.createData()
DAM.calc(DAD,x0,u0)
DAM.calcDiff(DAD,x0,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [178]:
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x0, u0)
DAMND.calcDiff(DADND, x0, u0)
NDFx = DADND.Fx
NDFu = DADND.Fu

In [179]:
np.allclose(myFx[:,18:],NDFx[:,18:],atol=1e-1)

False

In [139]:
np.allclose(myFx[:,:18],NDFx[:,:18],atol=1e-5)

False

In [180]:
np.allclose(myFu,NDFu,atol=1e-6)

False

### test 3: 静止, u=0.1

In [ ]:
import utils.models
import warnings
warnings.filterwarnings('ignore')
import importlib
importlib.reload(utils.models)
from utils.models import DAM_contact

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0)
x0 = rmodel.defaultState

u0 = np.ones(12)*0.1

DAD = DAM.createData()
DAM.calc(DAD,x0,u0)
DAM.calcDiff(DAD,x0,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [150]:
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x0, u0)
DAMND.calcDiff(DADND, x0, u0)
NDFx = DADND.Fx
NDFu = DADND.Fu

In [151]:
np.allclose(myFx[:,18:],NDFx[:,18:],atol=1e-6)

True

In [152]:
np.allclose(myFx[:,:18],NDFx[:,:18],atol=1e-5)

True

In [153]:
np.allclose(myFu,NDFu,atol=1e-6)

True

### test 4: 3 separate, u=0.1

In [ ]:
import utils.models
import warnings
warnings.filterwarnings('ignore')
import importlib
importlib.reload(utils.models)
from utils.models import DAM_contact

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0)
x0 = rmodel.defaultState

x0[8] = -3.14/2
x0[11] = -3.14/2

u0 = np.ones(12)*0.1
u0[8] = -10

DAD = DAM.createData()
DAM.calc(DAD,x0,u0)
DAM.calcDiff(DAD,x0,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [155]:
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x0, u0)
DAMND.calcDiff(DADND, x0, u0)
NDFx = DADND.Fx
NDFu = DADND.Fu

In [156]:
np.allclose(myFx[:,18:],NDFx[:,18:],atol=1e-6)

True

In [157]:
np.allclose(myFx[:,:18],NDFx[:,:18],atol=1e-5)

True

In [158]:
np.allclose(myFu,NDFu,atol=1e-6)

True

### test 5: separate, u=0.1, rho=0.1, 有rho梯度会不一样

In [ ]:
import utils.models
import warnings
warnings.filterwarnings('ignore')
import importlib
importlib.reload(utils.models)
from utils.models import DAM_contact

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0.1)
x0 = rmodel.defaultState

x0[8] = -3.14/2
u0 = np.ones(12)*0.1

DAD = DAM.createData()
DAM.calc(DAD,x0,u0)
DAM.calcDiff(DAD,x0,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [160]:
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x0, u0)
DAMND.calcDiff(DADND, x0, u0)
NDFx = DADND.Fx
NDFu = DADND.Fu

### test 6: 4 separate, u=0.1

In [ ]:
import utils.models
import warnings
warnings.filterwarnings('ignore')
import importlib
importlib.reload(utils.models)
from utils.models import DAM_contact, ContactModel

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0)
x0 = rmodel.defaultState
x00 = x0.copy()

x00[2] += 0.2
u0 = np.ones(12)*0.1

DAD = DAM.createData()
DAM.calc(DAD,x00,u0)
DAM.calcDiff(DAD,x00,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [165]:
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x00, u0)
DAMND.calcDiff(DADND, x00, u0)
NDFx = DADND.Fx
NDFu = DADND.Fu

In [166]:
np.allclose(myFx[:,18:],NDFx[:,18:],atol=1e-6)

True

In [167]:
np.allclose(myFx[:,:18],NDFx[:,:18],atol=1e-5)

True

In [168]:
np.allclose(myFu,NDFu,atol=1e-6)

True

### test 7: 4 separate, u=None

In [ ]:
import utils.models
import warnings
warnings.filterwarnings('ignore')
import importlib
importlib.reload(utils.models)
from utils.models import DAM_contact, ContactModel
from utils.costs import handstand
costs = handstand(rmodel,state,actuation)

DT = 0.025
contact_ids = [lf_id, rf_id, lh_id, rh_id,
               ]
friction = [0.9] * 4
# [lfcalf_id, rfcalf_id, lhcalf_id, rhcalf_id]
contact_model = ContactModel(state,contact_ids,friction)
DAM = DAM_contact(state, actuation, costs[0], contact_model, DT, 0)
x0 = rmodel.defaultState
x00 = x0.copy()

x00[2] += 0.2
u0 = None

DAD = DAM.createData()
DAM.calc(DAD,x00,u0)
DAM.calcDiff(DAD,x00,u0)
myFx = DAD.Fx
myFu = DAD.Fu

In [173]:
DAMND = crocoddyl.DifferentialActionModelNumDiff(DAM, True)
DADND = DAMND.createData()
DAMND.calc(DADND, x00)
DAMND.calcDiff(DADND, x00)
NDFx = DADND.Fx
NDFu = DADND.Fu

In [174]:
np.allclose(myFx[:,18:],NDFx[:,18:],atol=1e-6)

True

In [175]:
np.allclose(myFx[:,:18],NDFx[:,:18],atol=1e-5)

True

In [176]:
np.allclose(myFu,NDFu,atol=1e-6)

True